# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ART001-coder/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane: Refresh / Content Opportunity Scoring** (continuing from W02's `content_refresh_anonymized.csv`, same lane).


## 1. My rule and its reason codes

**The two signals I'm checking first, and why these two.**

FlyRank's live product computes hand-written flags on top of these exact kinds of signals —
staleness feeds the refresh flags, CTR-vs-position feeds the CTR-fix logic, and volume feeds
quick-win tagging (`skills/flyrank/flyrank-context/SKILL.md`). Before I lean on any of them in
my own rule, I check whether the signal actually behaves the way the story assumes, on *this*
data, with visible `n`.

- **Signal A — staleness** (`days_since_last_update` / `freshness_tier`) — the signal directly
  behind FlyRank's refresh flags. Claim: "older content decays more."
- **Signal B — visibility/volume** (`impressions_90d`) — the signal behind quick-win-style
  gating (is there enough real search demand hitting this page to make acting on it worth an
  editor's time?). Claim: "more impression volume means more actual clicks are being captured/lost."

Both signals feed my rule below, and both are flag-linked, so I'm not testing a hunch — I'm
checking the actual load-bearing assumptions before I code them into a score.

**A note on what I'm testing against:** `trend_direction` is never a *feature* in my rule or in
any model later (it's derived from `trend_pct`, which is the label trap called out in
`skills/flyrank/flyrank-data/SKILL.md`). Here in the audit, though, it's exactly the right thing
to check a signal *against* — the whole point of a signal check is "does X actually relate to a
real outcome?", and `trend_direction` is an already-observed outcome, not something I'm predicting
forward. It never enters `baseline_action_score`.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 160)

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)
print(df.shape)

# ---- Signal A: staleness -> does it actually track decline? ----
# Sample-size floor: no verdict from a bucket under ~50 rows (auditing-signals skill).
order = ['0-30', '31-90', '91-180', '181+']
df['freshness_tier'] = pd.Categorical(df['freshness_tier'], categories=order, ordered=True)

n_a = df.groupby('freshness_tier', observed=True).size()
down_rate_a = df.groupby('freshness_tier', observed=True)['trend_direction'].apply(lambda s: (s == 'down').mean())
signal_a_table = pd.DataFrame({'n': n_a, 'down_rate': down_rate_a.round(3)})
print("\nSignal A - staleness (freshness_tier) vs observed decline rate:")
print(signal_a_table)
print(f"\nAll buckets clear the n>=50 floor: {(signal_a_table['n'] >= 50).all()}")

(30000, 44)

Signal A - staleness (freshness_tier) vs observed decline rate:
                    n  down_rate
freshness_tier                  
0-30            20480      0.511
31-90             175      0.589
91-180           9171      0.611
181+              174      0.471

All buckets clear the n>=50 floor: True


**Signal A verdict: MIXED.**

The down-rate rises from 0.511 (0-30 days) to 0.589 (31-90) to 0.611 (91-180) — that part
matches the refresh-flag story: staler content trends down more. But the most-stale bucket
(181+, n=174, above the floor) drops back to 0.471 — *below* the freshest bucket. Staleness
is directionally right up to ~180 days and then reverses, so I can't use "older is always
worse" as written. This is exactly the kind of honest negative that changes the rule: instead
of copying the textbook `>= 180` threshold, my rule below uses `>= 90` — the zone the data
actually shows elevated risk in, not the zone that just sounds intuitive.

In [2]:
# ---- Signal B: visibility/volume -> does more impression volume mean more real clicks at stake? ----
bins = [0, 50, 500, 2000, 10000, 600000]
labels = ['0-50', '51-500', '501-2000', '2001-10000', '10000+']
df['impr_bucket'] = pd.cut(df['impressions_90d'], bins=bins, labels=labels, include_lowest=True)

signal_b_table = df.groupby('impr_bucket', observed=True).agg(
    n=('content_id', 'size'),
    mean_clicks=('clicks_90d', 'mean'),
    mean_ctr=('ctr', 'mean'),
).round(2)
print("Signal B - impression volume vs clicks actually captured:")
print(signal_b_table)
print(f"\nAll buckets clear the n>=50 floor: {(signal_b_table['n'] >= 50).all()}")

Signal B - impression volume vs clicks actually captured:
                n  mean_clicks  mean_ctr
impr_bucket                             
0-50         6528         0.12      1.40
51-500       6757         0.52      0.27
501-2000     6502         2.31      0.21
2001-10000   6611        13.90      0.29
10000+       3602       103.21      0.32

All buckets clear the n>=50 floor: True


**Signal B verdict: CONFIRMED.**

Mean clicks captured rises monotonically and sharply with impression volume: 0.12 → 0.52 →
2.31 → 13.90 → 103.21 clicks/page, across five buckets each with thousands of rows (n from
3,602 to 6,757 — well clear of the floor). This justifies gating my rule on `impressions_90d`:
below ~500 impressions a page is averaging under 1 click regardless of anything else, so there's
no real traffic at stake yet — acting on it wastes an editor's time. (Side note, not part of the
verdict: `mean_ctr` is highest in the lowest-impression bucket, 1.40% vs ~0.2-0.3% elsewhere —
almost certainly small-sample noise from pages with a handful of impressions, not a real CTR
advantage. I'm not building on that number.)

## My rule, in plain words

*A page is worth putting in the refresh queue if it hasn't been touched in at least 90 days
AND it's still pulling in enough impressions (500+) that a refresh would actually move real
traffic — the score is how many impressions are sitting behind that opportunity.*

That's the whole rule: `stale AND visible`, gated the way `skills/building-baselines/SKILL.md`
builds its reference example (`stale * visible * impressions`) — no fitted weights, three
sentences, readable by a non-engineer. Reason code and action are both single, fixed values,
not a menu of six like a full product rule would have — this is a baseline, meant to be
honestly beatable.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write `work/outputs/baseline_action_score.csv`.*

In [3]:
import os

# Rule inputs are both signals I just checked above — nothing future-window, nothing label-derived.
stale_flag = (df['days_since_last_update'] >= 90).astype(int)      # signal A, re-thresholded per the audit
visible_flag = (df['impressions_90d'] >= 500).astype(int)          # signal B

df['stale_flag'] = stale_flag
df['visible_flag'] = visible_flag
df['baseline_action_score'] = stale_flag * visible_flag * df['impressions_90d']

df['reason_code'] = np.where(
    (stale_flag == 1) & (visible_flag == 1), 'stale_visible_page', 'not_flagged'
)
df['action'] = np.where(
    (stale_flag == 1) & (visible_flag == 1), 'refresh', 'monitor'
)

queue_cols = [
    'content_id', 'client_id', 'baseline_action_score', 'reason_code', 'action',
    'days_since_last_update', 'freshness_tier', 'impressions_90d', 'clicks_90d',
    'avg_position', 'ctr', 'word_count', 'content_age_days', 'content_type', 'main_intent',
]
queue = df[queue_cols].sort_values('baseline_action_score', ascending=False).reset_index(drop=True)
queue.insert(0, 'rank', np.arange(1, len(queue) + 1))

n_flagged = int((queue['baseline_action_score'] > 0).sum())
base_rate = n_flagged / len(queue)
print(f"Total rows: {len(queue):,}")
print(f"Flagged (refresh): {n_flagged:,}  ({base_rate:.1%} of all pages)")
print(f"Monitor (everything else): {len(queue) - n_flagged:,}")

os.makedirs('../outputs', exist_ok=True)
OUT_PATH = '../outputs/baseline_action_score.csv'
queue.to_csv(OUT_PATH, index=False)
print(f"\nWrote: {OUT_PATH}")

queue.head(10)

Total rows: 30,000
Flagged (refresh): 6,575  (21.9% of all pages)
Monitor (everything else): 23,425



Wrote: ../outputs/baseline_action_score.csv


,rank,content_id,client_id,baseline_action_score,reason_code,action,days_since_last_update,freshness_tier,impressions_90d,clicks_90d,avg_position,ctr,word_count,content_age_days,content_type,main_intent
0,1,content_5fe46e04994d,client_4e07408562,517715,stale_visible_page,refresh,104,91-180,517715,741,4.2,0.14,NaN,537,keyword article,informational
1,2,content_2dba2b1f9536,client_6208ef0f77,443434,stale_visible_page,refresh,104,91-180,443434,910,27.9,0.21,7676.0,299,keyword article,informational
2,3,content_2c2606c5d176,client_19581e27de,347399,stale_visible_page,refresh,104,91-180,347399,1854,4.2,0.53,NaN,362,keyword article,commercial
3,4,content_cb112fce36be,client_19581e27de,309910,stale_visible_page,refresh,104,91-180,309910,492,5.6,0.16,2761.0,126,keyword article,transactional
4,5,content_9532f197bbc8,client_4e07408562,309192,stale_visible_page,refresh,104,91-180,309192,2689,2.0,0.87,NaN,445,keyword article,informational
5,6,content_36ff89c8214e,client_19581e27de,295097,stale_visible_page,refresh,104,91-180,295097,154,7.3,0.05,NaN,144,keyword article,informational
6,7,content_b28d1efd668f,client_6208ef0f77,286608,stale_visible_page,refresh,104,91-180,286608,169,26.2,0.06,6901.0,153,keyword article,transactional
7,8,content_813e88069237,client_6208ef0f77,233561,stale_visible_page,refresh,104,91-180,233561,129,26.2,0.06,4610.0,153,keyword article,commercial
8,9,content_c21024970297,client_19581e27de,211366,stale_visible_page,refresh,104,91-180,211366,870,5.1,0.41,2874.0,126,keyword article,commercial
9,10,content_c8e9d6ab9013,client_19581e27de,208678,stale_visible_page,refresh,104,91-180,208678,0,9.7,0.00,NaN,362,keyword article,informational


## 3. Top-10 review

*For each of the top ten: the action, why it's there, and what would make it wrong.*

Reading these by hand — the whole point of this step is to catch the rule being confidently
wrong, not to confirm it's right.

In [4]:
top10 = queue.head(10)[['rank', 'content_id', 'content_type', 'main_intent', 'baseline_action_score',
                          'impressions_90d', 'clicks_90d', 'avg_position', 'ctr', 'word_count',
                          'days_since_last_update', 'reason_code', 'action']]
top10

,rank,content_id,content_type,main_intent,baseline_action_score,impressions_90d,clicks_90d,avg_position,ctr,word_count,days_since_last_update,reason_code,action
0,1,content_5fe46e04994d,keyword article,informational,517715,517715,741,4.2,0.14,NaN,104,stale_visible_page,refresh
1,2,content_2dba2b1f9536,keyword article,informational,443434,443434,910,27.9,0.21,7676.0,104,stale_visible_page,refresh
2,3,content_2c2606c5d176,keyword article,commercial,347399,347399,1854,4.2,0.53,NaN,104,stale_visible_page,refresh
3,4,content_cb112fce36be,keyword article,transactional,309910,309910,492,5.6,0.16,2761.0,104,stale_visible_page,refresh
4,5,content_9532f197bbc8,keyword article,informational,309192,309192,2689,2.0,0.87,NaN,104,stale_visible_page,refresh
5,6,content_36ff89c8214e,keyword article,informational,295097,295097,154,7.3,0.05,NaN,104,stale_visible_page,refresh
6,7,content_b28d1efd668f,keyword article,transactional,286608,286608,169,26.2,0.06,6901.0,104,stale_visible_page,refresh
7,8,content_813e88069237,keyword article,commercial,233561,233561,129,26.2,0.06,4610.0,104,stale_visible_page,refresh
8,9,content_c21024970297,keyword article,commercial,211366,211366,870,5.1,0.41,2874.0,104,stale_visible_page,refresh
9,10,content_c8e9d6ab9013,keyword article,informational,208678,208678,0,9.7,0.00,NaN,104,stale_visible_page,refresh


**Top 10, one line each (action / why / what would make it wrong):**

1. **content_5fe46e0499** — `refresh` — 517,715 impressions, 104 days stale, position 4.2 but
   CTR only 0.14% at that position. — *Wrong if:* the low CTR is a title/meta problem, not a
   content-staleness problem — refreshing the body won't fix a snippet issue.
2. **content_2dba2b1f95** — `refresh` — 443,434 impressions, position 27.9, trend already
   `stable`. — *Wrong if:* it's stable because it's already found its ceiling at page 3 — a
   refresh may not move position at all without a bigger content or backlink change.
3. **content_2c2606c5d1** — `refresh` — 347,399 impressions, position 4.2, `word_count` missing
   (NaN — not zero, per the data dictionary's known content-type gap). — *Wrong if:* the missing
   word_count hides that this is actually a thin or malformed page, not a normal article.
4. **content_cb112fce36** — `refresh` — 309,910 impressions, transactional intent, trending
   `down`. — *Wrong if:* the decline is seasonal for this transactional query, not content
   decay — refreshing mid-season could be wasted effort.
5. **content_9532f197bb** — `refresh` — position 2.0 (near-top), CTR 0.87% (highest in the top
   10), still trending `down`. — *Wrong if:* this is a page that's fine and just riding a
   volatile SERP feature (e.g. a featured snippet flicker) — refreshing a healthy page wastes
   the slot a genuinely broken page could use.
6. **content_36ff89c821** — `refresh` — 295,097 impressions but only 154 clicks and CTR 0.05% —
   the weakest capture rate in the top 10 relative to its volume. — *Wrong if:* the query intent
   fundamentally mismatches the page (a CTR problem, not a staleness problem) — see section 4.
7. **content_b28d1efd66** — `refresh` — position 26.2 (page 3), trend `stable`, `word_count`
   6,901 (already long). — *Wrong if:* it's stable at page 3 because of thin topical authority,
   not thin content — a refresh of an already-long page may not be the highest-leverage fix.
8. **content_813e880692** — `refresh` — near-identical to #7 (position 26.2, low CTR 0.06%) but
   trending `down`. — *Wrong if:* both #7 and #8 share a client/template issue that a single
   page-level refresh can't fix (worth checking if they share `client_id`).
9. **content_c21024970** — `refresh` — position 5.1, CTR 0.41% (healthiest ratio in the top 10),
   trend `stable`. — *Wrong if:* this is genuinely healthy and just needs a lighter monitor
   touch, not a full refresh — it's the closest-to-fine page in the set.
10. **content_c8e9d6ab90** — `refresh` — 208,678 impressions but **0 clicks, CTR 0.00%**. —
    *Wrong if:* this is the single clearest weak pick in the top 10 — see section 4, it's flagged
    first there.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# ---- Weak pick, made concrete ----
weak = queue[queue['content_id'] == 'content_c8e9d6ab9013']
print("Weakest top-10 pick:")
print(weak[['rank', 'content_id', 'impressions_90d', 'clicks_90d', 'ctr', 'action']].to_string(index=False))

Weakest top-10 pick:
 rank           content_id  impressions_90d  clicks_90d  ctr  action
   10 content_c8e9d6ab9013           208678           0  0.0 refresh


**Weak pick, called out:** rank 10, `content_c8e9d6ab9013` — 208,678 impressions and
**zero** clicks in 90 days (`ctr = 0.00`). My score only looks at `impressions_90d`, so it
can't tell the difference between "used to convert, now stale" and "never converts, wrong query
match entirely." A page that's been showing up with zero clicks for months likely has a
title/snippet or search-intent problem the CTR-fix lane would catch — refreshing the *content*
of a page nobody clicks on is very likely the wrong first move. Rank 6 (`content_36ff89c8214e`,
154 clicks on 295K impressions, CTR 0.05%) has a milder version of the same issue. Both point at
the same rule gap: my score rewards raw impression volume without checking whether any of those
impressions are converting to clicks at all — that's the honest thing this baseline should lose
to a model on.

In [6]:
# ---- Leakage check ----
# 1. Confirm the rule's own inputs are only signal-A / signal-B columns, nothing else.
rule_inputs = {'days_since_last_update', 'impressions_90d'}
print("Rule inputs:", rule_inputs)

# 2. Confirm no product-flag columns exist in the source data to begin with (they're never shipped).
product_flags = {'health_score', 'priority_score', 'action_type', 'needs_ctr_fix', 'is_quick_win', 'refresh_tier'}
print("Product-flag columns present in source data:", product_flags & set(df.columns))  # must be empty

# 3. Confirm no future-window columns were used as rule inputs.
future_window_cols = {'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d',
                       'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d'}
print("Future/adjacent-window columns used in rule:", future_window_cols & rule_inputs)  # must be empty

# 4. Confirm no label-derived columns were used as rule inputs (the label trap from flyrank-data skill).
label_derived_cols = {'trend_direction', 'trend_pct', 'is_declining_label'}
print("Label-derived columns used in rule:", label_derived_cols & rule_inputs)  # must be empty

assert not (product_flags & set(df.columns)), "product flag leaked into source data"
assert not (future_window_cols & rule_inputs), "future window leaked into rule"
assert not (label_derived_cols & rule_inputs), "label-derived column leaked into rule"
print("\nLeakage check passed: rule uses only days_since_last_update and impressions_90d.")

Rule inputs: {'impressions_90d', 'days_since_last_update'}
Product-flag columns present in source data: set()
Future/adjacent-window columns used in rule: set()
Label-derived columns used in rule: set()

Leakage check passed: rule uses only days_since_last_update and impressions_90d.


**Leakage check result: clean.** The rule's only two inputs are `days_since_last_update` and
`impressions_90d` — both already-known, already-elapsed 90-day-window signals available before
any decision point, not future-window columns and not derived from `trend_direction` /
`trend_pct` / any label. Product-decision columns (`health_score`, `is_quick_win`, etc.) aren't
even shipped in this dataset (`docs/ml-intern-dataset-and-lane-guide.md`, section 4) — so there
was nothing to accidentally copy in the first place. `trend_direction` only appears above as an
*audit* check (section 1) and in the review columns for human context (section 3) — never inside
`baseline_action_score`.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Two signal verdicts with visible bucket tables and n (staleness = MIXED, impression
      volume = CONFIRMED; staleness is flag-linked to refresh flags, volume is flag-linked to
      quick-win gating)
- [x] One rule: score = `stale_flag * visible_flag * impressions_90d`, one reason code
      (`stale_visible_page` / `not_flagged`), one action label (`refresh` / `monitor`)
- [x] Ranked queue written from the notebook to `work/outputs/baseline_action_score.csv`
- [x] Ten reviewed rows, each with action / why / what would make it wrong
- [x] No future-window or label-derived columns used as rule inputs (checked in section 4)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.